# 15. Descriptors & Properties
Exhaustive guide to property getters/setters/deleters, descriptors validate protocols, __set_name__ names binding, and weak references.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for combined analysis questions at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Property getters decorators
**Explanation**: Virtual attribute properties using @property.

**Syntax**:
```python
@property
def virtual_attribute(self): return self._value
```



In [ ]:
class Box:
    @property
    def size(self): return 10
print(Box().size)

### 2. Property setters validation
**Explanation**: Validate properties on assignment.

**Syntax**:
```python
@virtual_attribute.setter
def virtual_attribute(self, value):
```



In [ ]:
class ValidatedTransaction:
    @property
    def balance(self): return self._balance
    @balance.setter
    def balance(self, value):
        if value < 0: raise ValueError('Min 0')
        self._balance = value
tx_instance = ValidatedTransaction()
tx_instance.balance = 5
print(tx_instance.balance)

### 3. Property deleters unbindings
**Explanation**: Clean up properties on deletion.

**Syntax**:
```python
@virtual_attribute.deleter
def virtual_attribute(self): del self._value
```



In [ ]:
class ValidatedTransaction:
    @property
    def balance(self): return self._balance
    @balance.deleter
    def balance(self): print('Deleter execution logs reached')
del ValidatedTransaction().balance

### 4. Data Descriptors protocol
**Explanation**: Implement __set__ to intercept attribute assignments.

**Syntax**:
```python
class DescriptorClass:
    def __set__(self, instance_obj, value): pass
```

**Visual Explanation (Data with Baraa Style)**:
```mermaid
graph TD
    get[Access instance.validated_limit] -->|triggers Descriptor __get__| res(Reads validated_limit)
```


In [ ]:
class PositiveIntegerValidator:
    def __set__(self, instance, value): instance._validated_value = value
class BoxContainer:
    validated_limit = PositiveIntegerValidator()
container = BoxContainer()
container.validated_limit = 5
print(container._validated_value)

### 5. Non-data Descriptors protocol
**Explanation**: Implement __get__ but omit __set__.

**Syntax**:
```python
class DescriptorClass:
    def __get__(self, instance_obj, owner_class): pass
```



In [ ]:
class NonDataDescriptor:
    def __get__(self, instance, owner): return 'ND'
class BoxContainer:
    validated_limit = NonDataDescriptor()
print(BoxContainer().validated_limit)

### 6. Descriptor deletion protocol
**Explanation**: Implement __delete__ to intercept attribute deletions.

**Syntax**:
```python
class DescriptorClass:
    def __delete__(self, instance_obj): pass
```



In [ ]:
class NonDataDescriptor:
    def __delete__(self, instance): print('Delete intercepted safely')
class BoxContainer:
    validated_limit = NonDataDescriptor()
container = BoxContainer()
del container.validated_limit

### 7. The descriptor parameter __set_name__
**Explanation**: Enables descriptors to auto-detect their class attribute names in Python 3.6+.

**Syntax**:
```python
def __set_name__(self, owner_class, attribute_name):
```



In [ ]:
class DescriptorClass:
    def __set_name__(self, owner, name): self.name = name
class BoxContainer:
    validated_limit = DescriptorClass()
print('Bound name:', BoxContainer.validated_limit.name)

### 8. Accessing instances attribute values inside descriptor classes
**Explanation**: Read/write instance variables directly using descriptors.

**Syntax**:
```python
getattr(instance_obj, self.name)
```



In [ ]:
class DescriptorClass:
    def __set_name__(self, owner, name): self.name = name
    def __get__(self, instance, owner): return instance.__dict__.get(self.name)
    def __set__(self, instance, value): instance.__dict__[self.name] = value
class BoxContainer:
    validated_limit = DescriptorClass()
container = BoxContainer()
container.validated_limit = 100
print(container.validated_limit)

### 9. Reusing validated descriptor attributes fields
**Explanation**: Apply a descriptor to validate multiple fields.

**Syntax**:
```python
class ClassName:
    field_one = Validator()
    field_two = Validator()
```



In [ ]:
class DescriptorClass:
    def __set__(self, instance, value):
        if value <= 0: raise ValueError('Min 1')
        instance._v = value
class BoxContainer:
    validated_limit = DescriptorClass()
print(BoxContainer)

### 10. Modifying class descriptors states
**Explanation**: Modifying class descriptors changes behavior dynamically.

**Syntax**:
```python
ClassName.descriptor_name = new_descriptor
```



In [ ]:
class BoxContainer: pass
print(BoxContainer)

### 11. Bypassing descriptors accesses
**Explanation**: Read raw descriptor objects directly from class dictionaries.

**Syntax**:
```python
ClassName.__dict__['descriptor_name']
```



In [ ]:
class DescriptorClass: pass
class BoxContainer:
    validated_limit = DescriptorClass()
print(type(BoxContainer.__dict__['validated_limit']))

### 12. Descriptor protocol parameters structures
**Explanation**: Understand descriptor method arguments.

**Syntax**:
```python
def __get__(self, instance_obj, owner_class): pass
```



In [ ]:
class DescriptorClass:
    def __get__(self, instance, owner): return instance, owner
class BoxContainer:
    validated_limit = DescriptorClass()
print(BoxContainer().validated_limit)

### 13. Weak references utilization inside descriptors
**Explanation**: Uses weakrefs to prevent instance memory leaks in key-value maps.

**Syntax**:
```python
import weakref
self.data = weakref.WeakKeyDictionary()
```



In [ ]:
import weakref
weak_map_dictionary = weakref.WeakKeyDictionary()
class DummyClass: pass
dummy_instance = DummyClass()
weak_map_dictionary[dummy_instance] = 'val'
print(weak_map_dictionary[dummy_instance])

### 14. Dynamic property creation
**Explanation**: Creates properties dynamically using the property() builtin.

**Syntax**:
```python
property_name = property(fget, fset)
```



In [ ]:
class BoxContainer:
    def __init__(self): self._size = 5
    def get_size(self): return self._size
    size = property(get_size)
print(BoxContainer().size)

### 15. Class-level property implementations
**Explanation**: Implement descriptors at metaclass level to enable class-level properties.

**Syntax**:
```python
# Metaclass properties structures
```



In [ ]:
class MetaclassProperty(type):
    @property
    def system_info(cls): return 'MetaInfo'
class BoxContainer(metaclass=MetaclassProperty): pass
print(BoxContainer.system_info)

## Section 3: Fintech Interview Questions

### Q1: Implement a custom descriptor `AmountValidator` using `__set_name__` that raises ValueError if transaction amount is negative.

In [ ]:
# Solution:
class AmountValidator:
    def __set_name__(self, owner, name):
        self.name = f'_{name}'
    def __get__(self, instance, owner):
        return getattr(instance, self.name, 0.0)
    def __set__(self, instance, value):
        if value < 0.0:
            raise ValueError('Amount cannot be negative')
        setattr(instance, self.name, value)

class Account:
    balance = AmountValidator()
    def __init__(self, val): self.balance = val
    
acc = Account(150.0)
print('Validated balance:', acc.balance)


### Q2: Show how weakref prevents memory leaks in descriptors by writing dummy validation mappings.

In [ ]:
# Solution:
import weakref
class WeakStore:
    def __init__(self):
        self.data = weakref.WeakKeyDictionary()
        
ws = WeakStore()
print('WeakKeyDictionary instantiated safely:', ws.data)
